# Plant Disease Classification
Classifying tomato, potato and pepper leaf diseases from the PlantVillage dataset with a frozen MobileNetV3Large base.

In [ ]:
import os
import random
import shutil
import time
import numpy as np
import tensorflow as tf
from keras import layers, models
from keras.applications import MobileNetV3Large
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import kagglehub

In [ ]:
path = kagglehub.dataset_download("emmarex/plantdisease")
directory = os.path.join(path, "PlantVillage")

## Looking at the data

In [ ]:
# the download has a duplicate PlantVillage folder inside it, so skip that
class_names = sorted([d for d in os.listdir(directory)
                      if os.path.isdir(os.path.join(directory, d)) and d != "PlantVillage"])

counts = {}
for c in class_names:
    files = [f for f in os.listdir(os.path.join(directory, c)) if f.lower().endswith((".jpg", ".jpeg"))]
    counts[c] = len(files)
    print(f"{c}: {len(files)}")
print("Total:", sum(counts.values()))

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(list(counts.keys()), list(counts.values()))
plt.title("Images per class (full dataset)")
plt.xlabel("Number of images")
plt.tight_layout()
plt.show()

## Subsample and split
Training on all 20,000 images is too slow on my laptop's CPU, so I take up to 400 images per class. 15% of each class goes into a test folder that the model never sees during training, and the rest is split 82/18 into train and validation (roughly 70/15/15 overall).

In [ ]:
PER_CLASS = 400
TEST_FRAC = 0.15
random.seed(42)

if os.path.exists("data"):
    shutil.rmtree("data")

for c in class_names:
    files = sorted([f for f in os.listdir(os.path.join(directory, c)) if f.lower().endswith((".jpg", ".jpeg"))])
    random.shuffle(files)
    files = files[:PER_CLASS]
    n_test = int(len(files) * TEST_FRAC)
    for i, f in enumerate(files):
        split = "test" if i < n_test else "train_val"
        os.makedirs(os.path.join("data", split, c), exist_ok=True)
        shutil.copy(os.path.join(directory, c, f), os.path.join("data", split, c, f))

for split in ["train_val", "test"]:
    total = sum(len(os.listdir(os.path.join("data", split, c))) for c in class_names)
    print(split, total)

In [ ]:
BATCH = 16
SIZE = (224, 224)

def data_loading(subset):
    dataset = image_dataset_from_directory(
        "data/train_val",
        labels = "inferred",
        label_mode = "categorical",
        validation_split = 0.18,
        subset = subset,
        shuffle = True,
        seed = 123,
        batch_size = BATCH,
        image_size = SIZE,
    )
    return dataset

train_ds = data_loading("training")
val_ds = data_loading("validation")

# test set is not shuffled so predictions line up with the labels
test_ds = image_dataset_from_directory(
    "data/test",
    labels = "inferred",
    label_mode = "categorical",
    shuffle = False,
    batch_size = BATCH,
    image_size = SIZE,
)
class_names = train_ds.class_names
num_classes = len(class_names)

In [ ]:
# one example of each class
rep_images = {c: None for c in class_names}
for img, label in train_ds.unbatch():
    c = class_names[np.argmax(label.numpy())]
    if rep_images[c] is None:
        rep_images[c] = img.numpy().astype("uint8")
    if all(v is not None for v in rep_images.values()):
        break

plt.figure(figsize=(12, 7))
for i, c in enumerate(class_names):
    plt.subplot(3, 5, i + 1)
    plt.imshow(rep_images[c])
    plt.title(c.replace("_", " "), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Model
Same setup as my cancer detection models: frozen MobileNetV3Large with ImageNet weights, then three dense layers and a softmax. MobileNetV3 in Keras rescales the pixels itself so the images go in as 0-255.

In [ ]:
def create_model(num_classes):
    pre_model = MobileNetV3Large(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    # freeze the base so only the new layers train
    pre_model.trainable = False

    x = pre_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dense(128, activation='relu')(x)
    output_layer = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=pre_model.input, outputs=output_layer)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.0001),
        metrics=['accuracy']
    )
    return model

tf.random.set_seed(42)
model = create_model(num_classes)
print("Trainable params:", sum(np.prod(w.shape) for w in model.trainable_weights))

In [ ]:
start = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=8, verbose=2)
print(f"Training took {(time.time() - start) / 60:.1f} minutes")

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()
plt.show()

## Test set results

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Test loss: {loss:.4f}, Test accuracy: {accuracy:.4f}")

In [ ]:
y_true = np.concatenate([np.argmax(y, axis=1) for x, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
short = [c.replace("Tomato_", "T ").replace("Potato___", "Po ").replace("Pepper__bell___", "Pe ").replace("_", " ") for c in class_names]
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=short, yticklabels=short)
plt.title("Confusion matrix (test set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
os.makedirs("models", exist_ok=True)
model.save("models/plant_disease_model.keras")
print(f"Model size: {os.path.getsize('models/plant_disease_model.keras') / 1e6:.1f} MB")